# Real Estate Data Analysis

In this project, I will analyze the relationship between socioeconomic, demographic, and geographic characteristics and house values in California.

It involves combining **exploratory data analysis** and **machine learning** to investigate the factors associated with **real estate prices** and evaluate whether these characteristics can be used to **predict house values**.

I will use the following the pipeline below:

1. **Understand the dataset.**
2. **Explore the data to analyse and identify relevant patterns.**
3. Prepare the data for **machine learning.**
4. **Split the data into training and testing sets.**
5. **Establish a baseline for models evaluation.**
6. **Train and compare machine learning models.**
7. **Evaluation of the results.**
8. **Identification of the most relevant features for the best-performing model.**
9. **Discuss the results and limitations.**

## Importing the libraries

In [1]:
# Data manipulation and visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modelling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

## Data Loading

The California Housing dataset will be loaded using Scikit-learn. The dataset contains information about housing districts in California and will be used throughout the analysis and modelling stages of this project.

In [2]:
from sklearn.datasets import fetch_california_housing

In [3]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


# Preparing data for Machine Learning

Based on my exploratory data analysis, how can we use these characteristics we've analysed and investigated to estimate property values?

Before training any ML model to answer this, let's prepare the data understanding a fundamental structure:

Features (`x`) → Model (Actual value (`y`) → Comparison) → Prediction (`ŷ`)

Value we want to predict (`y`): `MedHouseVal`

## Data Separation

Before even training the models, we must separate the target variable from the features used to make the predictions.

In [4]:
x = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

The remaining variables will be used as input features for the models.

In [5]:
x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


Now, let's start answering:
- Is it possible to estimate property values ​​using the available features?

To do this, we've splitted the DataFrame.
- (`y`) target: `MedHouseVal`
- (`x`) information used to make the prediction (basically everything except `y`).

### Train-Test Split

The data must be divided into training and testing sets to evaluate how well the models perform. Bou could might think: "I have 20,640 rows in the dataset. Why not just feed them all to the model to learn from?"

The problem is that we need to determine whether the model has truly learned to "generalize" or has simply become good at predicting the data it has already seen. For example:

Imagine a student studying for an exam. Suppose the teacher assigns 100 practice problems, and the student studies all of them.

Later, the teacher gives the student the exact same 100 problems, and the student gets 100% of them right.

* Question: Can I conclude that the student has actually learned the material?
* Answer: Not necessarily. They might have simply memorized the answers.

Now imagine this scenario: The student has 100 problems to study, but the actual exam contains different problems.

So, if the student performs well on the new exam, we have much stronger evidence that they have truly learned the content. Machine learning works in a similar way. 

The model can only learn from the training data (approximately 80% of the data available of the dataset). We then use data it never saw during training to evaluate its ability to make predictions.

In [7]:
# Train/Test Split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2, # set aside 20% of the data for testing. Consequently, 80% (training and 20%) → testing.
    random_state=42 # Fixed in 42 to make the experiment reproducible.
)

print(f"Training data: {x_train.shape}")
print(f"Test data: {x_test.shape}")

Training data: (16512, 8)
Test data: (4128, 8)


This split (training and testing) is not just a formality. We are trying to answer two different questions:

Training:
"Can the model learn a relationship between the features and the price?"

Testing:
"After learning, can it use that relationship to predict prices it has never seen before?"

This second question is fundamental. Because the goal of machine learning is not to memorize our 20,640 records; it is to learn patterns that can be applied to new data.

## Modeling

### Baseline

Before training machine learning models, a baseline is established as a reference for evaluating model performance.

What's a baseline? A baseline is a simple solution that serves as a performance benchmark for our model; the goal is for our model to outperform this simple solution.

This answers the question:
"Can a Machine Learning model actually perform better than an extremely simple strategy?"

In our case, we will use a simple strategy: always predicting the average value of the properties found in the training data. For example:

Imagine these are the property values:
* 100
* 150
* 200
* 250
* 300

The average value is: **200**

So, our baseline could simply predict these values ​​for the properties:
* 200
* 200
* 200
* 200
* 200

Our baseline does not know about:
- income;
- location;
- house age;
- number of rooms.

This baseline simply uses the average as the prediction for everything.

Naturally, we don't expect this to be an excellent strategy, but we now have a benchmark comparisson.

An important note: the baseline can also only learn from the training data, but I will not calculate the average using the entire dataset because it would be using information from the test set.

Remember:

Training → the model can know this

Test → the model cannot know this

In [8]:
baseline_prediction = y_train.mean()

print(f"Previsão da baseline: {baseline_prediction:.2f}")
# This value represents the average of the property values ​​present only in the training set.

Previsão da baseline: 2.07


But we still need to answer: What is the prediction error of this baseline?

I'll start by calculating the mean absolute error (MAE).

The idea is simple.
For each prediction:

* Real Value:     3.00
* Prediction:     2.00
* Difference:     1.00

For another record:

* Real Value:     1.50
* Prediction:     2.00
* Difference:     0.50

I will calculate the absolute values ​​of these differences and then take the average. Therefore: lower MAE = better.

To do this, I will import the following metric from scikit-learn to generate a prediction for each test record: MAE (mean_absolute_error).

The baseline always predicts the same value, so:

In [9]:
from sklearn.metrics import mean_absolute_error # MAE
baseline_predictions = [baseline_prediction] * len(y_test)
# This creates a list with the repeated average for each property in the test set.
# Now we calculate:
baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)
print(f"Baseline MAE: {baseline_mae:.2f}") 

MAE da baseline: 0.91


The baseline achieved an MAE of approximately **0.91**, corresponding to an average prediction error of approximately **US$91,000**.

This is our first numerical baseline.
With this numerical baseline, from now on we can compare:

Baseline:

MAE = X

Model 1:

MAE = ?

Model 2:

MAE = ?

And then answer objectively:

Did the model actually showed a improve compared to a simple strategy as the baseline?

### First model: Linear Regression

I will use Linear Regression as the first machine learning model to determine whether the relationships between the features and house values can improve predictions compared to the baseline.

The idea behind Linear Regression is to try to find relationships between our features and the values we want to predict.

Structure: 
x_train → input information
y_train → correct answers
  ↓
model
  ↓
learns patterns and relationships

The model receives the training data and tries to find relationships between the characteristics of x_train and the values ​​of y_train (I do not use the nomenclatures x and y instead of x_train and y_train because we want to preserve the test data for an honest evaluation.).

In [10]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train, y_train) # Training
y_pred = model.predict(x_test)
# The model receives x_test and generates y_pred, characteristics of features it never saw during training.

The Linear Regression model achieved an MAE of approximately **0.53**, reducing the prediction error compared to the baseline.